In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

In [212]:
s = np.load('fluxes_202505fdt.npz')
flux_density_1 = s['flux_density']
variance_1 = s['variance']
lat_1 = s['lat'].astype(float)

dlat_1 = lat_1[1] - lat_1[0]
lat_1 += dlat_1 / 2

In [213]:
s = np.load('fluxes_202509fdt.npz')
flux_density_2 = s['flux_density']
variance_2 = s['variance']
lat_2 = s['lat'].astype(float)

dlat_2 = lat_2[1] - lat_2[0]
lat_2 += dlat_2 / 2

In [270]:
fig = plt.figure(figsize=(10,8))
plt.errorbar(lat_1, flux_density_1, yerr=np.sqrt(variance_1))
plt.errorbar(lat_2, flux_density_2, yerr=np.sqrt(variance_2))

plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [246]:
def diffuse(y, qi):
    from scipy.linalg import solve_banded
    
    V = - qi[:-1] / 2
    U = - qi[1:] / 2
    D = 1 + (qi[:-1] + qi[1:]) / 2
    B = y + (qi[1:] * (np.roll(y, -1) - y) - qi[:-1] * (y - np.roll(y, 1))) / 2

    U[1] = 0#-1
    V[-2] = 0#-1
    D[[0,-1]] = 1
    B[[0,-1]] = y[[0,-1]]#0
    
    return solve_banded((1, 1), [U, D, V], B, True, True)
    


In [267]:
R = 695e8
u = 900
d = 1e12

dx = dlat_1 * np.pi / 180 * R
dt = 24 * 3600 

xi = np.arange(-90,90.5)
qi = d * np.cos(xi * np.pi / 180) * dt / dx ** 2
vi = u * np.sin(2 * xi * np.pi / 180) * np.cos(xi * np.pi / 180) * dt / dx
Si = R ** 2 * (1 - np.sin(xi * np.pi / 180)) * 2 * np.pi

dS = Si[1:] - Si[:-1]
y = flux_density_1 * dS

for _ in range(4 * 30):
    F = np.where(vi > 0, vi * np.append(y, 0), vi * np.append(0, y))
    dy = F - np.roll(F, 1)
    y = y - dy[:-1]
    y = diffuse(y, qi)

y = y / dS

In [271]:
x = np.arange(-90,90) + 0.5

plt.figure(figsize=(10,8))
plt.plot(x, flux_density_1)
plt.plot(x, y)
plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [269]:
t = np.where(x > 65)

np.trapz(y[t] * dS[t]), np.trapz(flux_density_2[t] * dS[t]), 

(7.280620625716784e+21, 7.340123218119355e+21)